# Liver lobule analysis
### For Natalie Porat-Shliom lab

### 12/07/23
### Author: Andy D. Tran, CCR Microscopy Core, LCBG, CCR, NCI

### Use Cellpose models for cell, mitochondria, and lipid droplet segmentation. Generate EDT maps for
### user-defined central and portal veins. Output intensity, geometric and spatial parameters.

### ----------------------------------------------------------------------------
### Navigation

[Segmentation and quantification](#main_function)
    
[Load existing segmentation](#alt_function)

[Quantification](#quantification)


In [ ]:
# Import libraries--------------------------------------------------------------------------------
import os 
import numpy as np 
from numpy.typing import NDArray
import cv2
import pandas as pd 
import napari 
from napari.layers import Labels
import re

from scipy import ndimage as nd 
from tifffile import imread, imwrite
from skimage.measure import regionprops
from tqdm import tqdm
from cellpose import models, io

from pathlib import Path
from dotenv import load_dotenv
from loguru import logger
from typing import Any

try:
    logger.remove(0)
    logger.add(lambda msg: tqdm.write(msg, end=""), colorize=True)
except Exception as e:
    pass

In [2]:
# config project --------------------------------------------------------------------------------
# Load environment variables from .env file if it exists
load_dotenv()
# Paths
PROJ_ROOT: Path = Path("notebooks").resolve().parents[0].parents[0]
logger.info(f"PROJ_ROOT path is: {PROJ_ROOT}")

DATA_DIR: Path = PROJ_ROOT / "data"
RAW_DATA_DIR: Path = DATA_DIR / "raw"
IMAGES_DIR: Path = DATA_DIR / "images"
INTERIM_DATA_DIR: Path = DATA_DIR / "interim"
PROCESSED_DATA_DIR: Path = DATA_DIR / "processed"
EXTERNAL_DATA_DIR: Path = DATA_DIR / "external"

MODELS_DIR: Path = PROJ_ROOT / "models"

REPORTS_DIR: Path = PROJ_ROOT / "reports"
FIGURES_DIR: Path = REPORTS_DIR / "figures"

2026-09-07 16:41:04.192 | INFO     | __main__:<module>:6 - PROJ_ROOT path is: E:\Scripts\scPhenomics


In [ ]:
# Define functions---------------------------------------------------------------------------------
def model_apply(model: models.CellposeModel, img: NDArray) -> NDArray | Any:
    """Apply the Cellpose model to an image.
    You will get a tuple containing (masks, flows, styles) as output.
    The mask is the most important output,
    which contains the segmented objects in the image.
    """
    mask, _, _ = model.eval(img, channels = [0, 0])
    
    return mask

def object_quant(lbl_img, lipid_img) -> pd.DataFrame:
    """
    Quantify mean lipid intensity for each object in a labeled image.

    This function computes the mean intensity of the lipid channel for every
    segmented region (object) in the input label image. It is typically used
    as a precursor to filtering objects based on lipid signal intensity.

    Parameters
    ----------
    lbl_img : np.ndarray
        2D labeled image where each unique positive integer represents a
        distinct segmented object (e.g., lipid droplets).
    lipid_img : np.ndarray
        2D grayscale image corresponding to the lipid fluorescence channel.
        Must have the same spatial dimensions as lbl_img.

    Returns
    -------
    pd.DataFrame
        A DataFrame with the following columns:
        - 'roi_id' : int, the unique label of each object.
        - 'lipid_int' : float, the mean pixel intensity of the lipid channel
          within the object's region.
    """
    df = pd.DataFrame() 
    
    roi_id = [] 
    lipid_int = [] 
    
    roiprops = regionprops(lbl_img, intensity_image = lipid_img)
    
    for roi in tqdm(range(len(roiprops))):
        roi_id.append(roiprops[roi].label)
        lipid_int.append(roiprops[roi].mean_intensity)
        
    df['roi_id'] = roi_id
    df['lipid_int'] = lipid_int
    
    return df

def object_filter(lbl_img: NDArray, int_df: pd.DataFrame, int_thresh: int) -> NDArray:
    """
    Filter labeled objects based on a lipid intensity threshold.

    This function removes objects whose mean lipid intensity falls below a
    specified threshold by setting their pixel values to zero. It preserves
    the original labels for objects that pass the filter.

    Parameters
    ----------
    lbl_img : np.ndarray
        2D labeled image with integer labels for each object.
    int_df : pd.DataFrame
        DataFrame containing object intensity data. Must include a column
        named 'roi_id' (object labels) and a column named 'lipid_int'
        (mean lipid intensity for each object).
    int_thresh : int
        Intensity threshold. Objects with mean lipid intensity less than or
        equal to this value will be removed (set to 0).

    Returns
    -------
    np.ndarray
        A new labeled image of the same shape as lbl_img, where objects
        with lipid_int <= int_thresh have been set to 0. Objects that pass
        the threshold retain their original labels.
    """
    roi_id = lbl_img.reshape(-1)
    
    mask_df = pd.DataFrame()
    mask_df['roi_id'] = roi_id
    mask_df = pd.merge(mask_df, int_df, how = 'left', on = ['roi_id'])
    
    mask_tmp = mask_df['lipid_int'].to_numpy()
    mask_tmp = mask_tmp.reshape(lbl_img.shape)
    
    mask = np.where(mask_tmp <= int_thresh, 0, lbl_img)
    
    return mask

def roi_quant(
        lbl_img: NDArray, cell_img: NDArray, mito_img: NDArray, lipid_img: NDArray, 
        cv_edt: NDArray, pv_edt: NDArray, organelle_edt: NDArray
        ) -> pd.DataFrame:
    """
    Extract comprehensive morphometric, intensity, and spatial features for
    each object in a labeled image.

    This function computes a wide range of quantitative features for each
    segmented object (e.g., a cell) using the input label image and
    corresponding fluorescence and distance transform maps.

    Parameters
    ----------
    lbl_img : np.ndarray
        2D labeled image where each unique positive integer represents a
        distinct segmented object (e.g., a cell).
    cell_img : np.ndarray
        2D image used for identifying cell boundaries (e.g., actin channel).
        Used to compute intensity features for each object.
    mito_img : np.ndarray
        2D fluorescence image of mitochondria. Used to compute mean
        mitochondrial intensity within each object.
    lipid_img : np.ndarray
        2D fluorescence image of lipid droplets. Used to compute mean
        lipid intensity within each object.
    cv_edt : np.ndarray
        2D Euclidean Distance Transform (EDT) map from the central vein (CV).
        Pixel values represent distance to the nearest CV boundary.
    pv_edt : np.ndarray
        2D Euclidean Distance Transform (EDT) map from the portal vein (PV).
        Pixel values represent distance to the nearest PV boundary.
    organelle_edt : np.ndarray
        2D Euclidean Distance Transform (EDT) map from an organelle of
        interest (e.g., mitochondria or lipid droplets). Pixel values
        represent distance to the nearest organelle surface.

    Returns
    -------
    pd.DataFrame
        A DataFrame with one row per object (ROI) containing the following
        feature groups:

        **Identification and Morphometrics:**
        - 'roi_id' : int, unique label of the object.
        - 'cell_id' : int, rounded mean intensity from cell_img (used as
          a secondary identifier).
        - 'area' : int, number of pixels in the object.
        - 'perimeter' : float, perimeter length of the object.
        - 'eccentricity' : float, eccentricity of the fitted ellipse
          (0 = circle, closer to 1 = elongated).
        - 'solidity' : float, ratio of area to convex hull area
          (1 = perfectly convex).
        - 'axis_major_length' : float, major axis length of fitted ellipse.
        - 'axis_minor_length' : float, minor axis length of fitted ellipse.
        - 'feret_diameter_max' : float, maximum Feret diameter (caliper
          distance).
        - 'centroid_x' : float, X-coordinate (column) of the object centroid.
        - 'centroid_y' : float, Y-coordinate (row) of the object centroid.

        **Intensity Features:**
        - 'mito_int' : float, mean mitochondrial fluorescence intensity.
        - 'lipid_int' : float, mean lipid fluorescence intensity.

        **Spatial Features (Central Vein):**
        - 'cv_min_dist' : float, minimum distance from object to CV.
        - 'cv_max_dist' : float, maximum distance from object to CV.
        - 'cv_mean_dist' : float, mean distance from object to CV.

        **Spatial Features (Portal Vein):**
        - 'pv_min_dist' : float, minimum distance from object to PV.
        - 'pv_max_dist' : float, maximum distance from object to PV.
        - 'pv_mean_dist' : float, mean distance from object to PV.

        **Spatial Features (Organelle Proximity):**
        - 'organelle_min_dist' : float, minimum distance from object to
          the nearest organelle.
        - 'organelle_max_dist' : float, maximum distance from object to
          the nearest organelle.
        - 'organelle_mean_dist' : float, mean distance from object to the
          nearest organelle.

    Notes
    -----
    This function uses scikit-image's `regionprops` internally and iterates
    over all objects. For large images with many objects, this operation
    can be time-consuming. Progress is reported via a tqdm progress bar.
    """
    df = pd.DataFrame() 
    
    roi_id = [] 
    cell_id = [] 
    area = [] 
    perimeter = []
    eccentricity = [] 
    solidity = [] 
    axis_major_length = []
    axis_minor_length = [] 
    feret_diameter_max = [] 
    centroid_x = [] 
    centroid_y = [] 
    mito_int = [] 
    lipid_int = [] 
    cv_min_dist = [] 
    cv_max_dist = [] 
    cv_mean_dist = [] 
    pv_min_dist = [] 
    pv_max_dist = [] 
    pv_mean_dist = []
    organelle_min_dist = []
    organelle_max_dist = [] 
    organelle_mean_dist = [] 
    
    cellprops = regionprops(lbl_img, intensity_image = cell_img)
    mitoprops = regionprops(lbl_img, intensity_image = mito_img)
    lipidprops = regionprops(lbl_img, intensity_image = lipid_img)
    cvprops = regionprops(lbl_img, intensity_image = cv_edt)
    pvprops = regionprops(lbl_img, intensity_image = pv_edt)
    organelleprops = regionprops(lbl_img, intensity_image = organelle_edt)
    
    for roi in tqdm(range(len(cellprops))):
        roi_id.append(cellprops[roi].label)
        cell_id.append(round(cellprops[roi].intensity_mean))
        area.append(cellprops[roi].area)
        perimeter.append(cellprops[roi].perimeter)
        eccentricity.append(cellprops[roi].eccentricity)
        solidity.append(cellprops[roi].solidity)
        axis_major_length.append(cellprops[roi].axis_major_length)
        axis_minor_length.append(cellprops[roi].axis_minor_length)
        feret_diameter_max.append(cellprops[roi].feret_diameter_max)
        centroid_x.append(cellprops[roi].centroid[1])
        centroid_y.append(cellprops[roi].centroid[0])
        mito_int.append(mitoprops[roi].intensity_mean)
        lipid_int.append(lipidprops[roi].intensity_mean)
        cv_min_dist.append(cvprops[roi].intensity_min)
        cv_max_dist.append(cvprops[roi].intensity_max)
        cv_mean_dist.append(cvprops[roi].intensity_mean)
        pv_min_dist.append(pvprops[roi].intensity_min)
        pv_max_dist.append(pvprops[roi].intensity_max)
        pv_mean_dist.append(pvprops[roi].intensity_mean)
        organelle_min_dist.append(organelleprops[roi].intensity_min)
        organelle_max_dist.append(organelleprops[roi].intensity_max)
        organelle_mean_dist.append(organelleprops[roi].intensity_mean)
        
    df['roi_id'] = roi_id
    df['cell_id'] = cell_id
    df['area'] = area
    df['perimeter'] = perimeter
    df['eccentricity'] = eccentricity
    df['solidity'] = solidity
    df['axis_major_length'] = axis_major_length
    df['axis_minor_length'] = axis_minor_length
    df['feret_diameter_max'] = feret_diameter_max
    df['centroid_x'] = centroid_x
    df['centroid_y'] = centroid_y
    df['mito_int'] = mito_int
    df['lipid_int'] = lipid_int
    df['cv_min_dist'] = cv_min_dist
    df['cv_max_dist'] = cv_max_dist
    df['cv_mean_dist'] = cv_mean_dist
    df['pv_min_dist'] = pv_min_dist
    df['pv_max_dist'] = pv_max_dist
    df['pv_mean_dist'] = pv_mean_dist
    df['organelle_min_dist'] = organelle_min_dist
    df['organelle_max_dist'] = organelle_max_dist
    df['organelle_mean_dist'] = organelle_mean_dist
    
    return df 

### Segmentation and quantification mode

<a id='main_function'></a>

### Set paths

<a id='set_path'></a>

In [4]:
# Define model paths--------------------------------------------------------------------------------------

cell_model_path: Path = MODELS_DIR / "brownl_cell_01"
mito_model_path: Path = MODELS_DIR / "brownl_mito_03"
lipid_model_path: Path = MODELS_DIR / "brownl_lipid_01"

logger.info(f"""
    Cell model path: {cell_model_path}
    Mito model path: {mito_model_path}
    Lipid model path: {lipid_model_path}
"""
)

2026-09-07 16:41:18.038 | INFO     | __main__:<module>:7 - 
    Cell model path: E:\Scripts\scPhenomics\models\brownl_cell_01
    Mito model path: E:\Scripts\scPhenomics\models\brownl_mito_03
    Lipid model path: E:\Scripts\scPhenomics\models\brownl_lipid_01



In [5]:
# Define file paths--------------------------------------------------------------------------

# input_path = input('Input path: ')
# output_path = input('Output path: ')

input_path: Path = IMAGES_DIR / "PLIN5 variants on CNTR diet" / "PLIN5 (1-424)"
output_path: Path = PROCESSED_DATA_DIR / "PLIN5 (1-424)"

logger.info(f"""
    Input path: {input_path}
    Output path: {output_path}
"""
)

2026-09-07 16:41:23.261 | INFO     | __main__:<module>:9 - 
    Input path: E:\Scripts\scPhenomics\data\images\PLIN5 variants on CNTR diet\PLIN5 (1-424)
    Output path: E:\Scripts\scPhenomics\data\processed\PLIN5 (1-424)



In [6]:
# input_path = os.path.normpath(input_path)
# output_path = os.path.normpath(output_path)

# cell_model_path = os.path.normpath(cell_model_path)
# mito_model_path = os.path.normpath(mito_model_path)
# lipid_model_path = os.path.normpath(lipid_model_path)

# GPU加速：关闭
cell_model = models.CellposeModel(gpu = False, pretrained_model = str(cell_model_path))# pyright: ignore[reportArgumentType]
mito_model = models.CellposeModel(gpu = False, pretrained_model = str(mito_model_path))# pyright: ignore[reportArgumentType]
lipid_model = models.CellposeModel(gpu = False, pretrained_model = str(lipid_model_path))# pyright: ignore[reportArgumentType]

### Load image

<a id='load_cell'></a>

In [7]:
# Load image-----------------------------------------------------------------------------------------------
tmp_list: list[Path] = list(Path(input_path).iterdir())

img_list: list[Path] = []
for tmp in tmp_list:
    if re.search('.tif', str(tmp)):
        img_list.append(tmp)
        
logger.info("images found:\n"+"\n".join(map(lambda x: str(x.name), img_list)))
logger.info(f"Number of images found: {len(img_list)}")
# img_index = int(input('Select image: '))
img_index: int = int("12")
img_index: int = int(img_index) - 1
logger.info(f"Selected image: {img_list[img_index].name}")

2026-09-07 16:41:30.472 | INFO     | __main__:<module>:9 - images found:
3916C M1_L1_Cropped PP-PC Axis 1.tif
3916C M1_L1_Cropped PP-PC Axis 2.tif
3916C M1_L2_Cropped PP-PC Axis 3.tif
3916C M1_L2_Cropped PP-PC Axis 4.tif
3916C M1_L2_Cropped PP-PC Axis 5.tif
3916C M1_L3_Cropped PP-PC Axis 6.tif
3917A M2_L1_Cropped PP-PC Axis 1.tif
3917A M2_L1_Cropped PP-PC Axis 2.tif
3917A M2_L2_Cropped PP-PC Axis 3.tif
3917A M2_L2_Cropped PP-PC Axis 4.tif
3917A M2_L3_Cropped PP-PC Axis 5.tif
3917A M2_L3_Cropped PP-PC Axis 6.tif
2026-09-07 16:41:30.473 | INFO     | __main__:<module>:10 - Number of images found: 12
2026-09-07 16:41:30.474 | INFO     | __main__:<module>:14 - Selected image: 3917A M2_L3_Cropped PP-PC Axis 6.tif


In [ ]:
# show image --------------------------------------------------------------------------------
img_path: Path = img_list[img_index] # img_list have full path
logger.info(f"Image path: {img_path}")
img = imread(img_path)

logger.info(img.shape)

viewer = napari.Viewer()
viewer.add_image(img, name='Original Image')

2026-09-07 19:03:09.743 | INFO     | __main__:<module>:3 - Image path: E:\Scripts\scPhenomics\data\images\PLIN5 variants on CNTR diet\PLIN5 (1-424)\3917A M2_L3_Cropped PP-PC Axis 6.tif
(3, 1444, 5188)


<Image layer 'Original Image' at 0x1806d5134d0>

In [9]:
# add channel images to viewer ---------------------------------------------------------------
# 正确分辨通道是模型正确运行的基础，FIJI分离通道观察形状以辅助区分：Image > Color > Split Channels
# actin_ch_number = int(input('Input actin channel: ')) - 1
# mito_ch_number = int(input('Input mitochondria channel: ')) - 1
# lipid_ch_number = int(input('Input lipid channel: ')) - 1
actin_ch_number = int("3") - 1
mito_ch_number = int("1") - 1
lipid_ch_number = int("2") - 1

actin_img = img[actin_ch_number, :, :]
mito_img = img[mito_ch_number, :, :]
lipid_img = img[lipid_ch_number, :, :]

viewer.add_image(actin_img, name="actin_img")
viewer.add_image(mito_img, name="mito_img")
viewer.add_image(lipid_img, name="lipid_img")

<Image layer 'lipid_img' at 0x1806bbfafd0>

In [10]:
viewer.close()

### Segmentation--------------------------------------------------------

###### Cell segmentation

In [11]:
mito_tmp = nd.gaussian_filter(mito_img, sigma = 50)
mito_tmp = mito_tmp * 3

tmp_img = np.add(actin_img, mito_tmp)

cell_mask = model_apply(cell_model, tmp_img)
cell_mask = np.uint16(cell_mask)

e:\Scripts\scPhenomics\.venv\Lib\site-packages\torch\jit\_script.py:365: FutureWarning: `torch.jit.script_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [12]:
viewer = napari.Viewer()
viewer.add_image(actin_img, colormap = 'red', blending = 'additive')
mito_layer = viewer.add_image(mito_img, colormap = 'green', blending = 'additive')
cell_layer = viewer.add_layer(Labels(cell_mask))

In [13]:
cell_mask = viewer.layers['cell_mask'].data
cell_mask = np.uint16(cell_mask)

In [15]:
cell_mask_path = output_path / (img_list[img_index].name.replace('.tif', '_cell.tif'))
imwrite(cell_mask_path, cell_mask)

viewer.close()

In [746]:
#cell_mask_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_cell.tif'))
#
#cell_mask = imread(cell_mask_path)

###### Mitochondria segmentation

In [16]:
mito_mask = model_apply(mito_model, mito_img)
mito_mask = np.uint16(mito_mask)

In [17]:
viewer = napari.Viewer()
viewer.add_image(mito_img, colormap = 'green', blending = 'additive')
mito_layer = viewer.add_layer(Labels(mito_mask))

In [18]:
mito_mask_path = output_path / (img_list[img_index].name.replace('.tif', '_mito.tif'))
imwrite(mito_mask_path, mito_mask)

viewer.close()

In [750]:
#mito_mask_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_mito.tif'))
#
#mito_mask = imread(mito_mask_path)

###### Lipid segmentation

In [19]:
lipid_mask_tmp = model_apply(lipid_model, lipid_img)
lipid_mask_tmp = np.uint16(lipid_mask_tmp)

lipid_df_tmp = object_quant(lipid_mask_tmp, lipid_img)
lipid_mask = object_filter(lipid_mask_tmp, lipid_df_tmp, 10)

  0%|          | 0/2686 [00:00<?, ?it/s]C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:22: FutureWarning: `RegionProperties.mean_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_mean` instead. 
  lipid_int.append(roiprops[roi].mean_intensity)
100%|██████████| 2686/2686 [00:00<00:00, 6862.49it/s]


In [20]:
viewer = napari.Viewer()
viewer.add_image(lipid_img, colormap = 'magenta', blending = 'additive')
lipid_layer = viewer.add_layer(Labels(lipid_mask))

In [21]:
lipid_mask = viewer.layers['lipid_mask'].data

In [22]:
lipid_mask_path = output_path / (img_list[img_index].name.replace('.tif', '_lipid.tif'))
imwrite(lipid_mask_path, lipid_mask)

viewer.close()

In [707]:
#lipid_mask_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_lipid.tif'))

#lipid_mask = imread(lipid_mask_path)

### Organelle overlap----------------------------------------

In [23]:
mito_binary = np.where(mito_mask > 0, 1, 0)
lipid_binary = np.where(lipid_mask > 0 , 1, 0)

overlap_tmp = np.add(mito_binary, lipid_binary)
overlap_mask = np.where(overlap_tmp == 2, cell_mask, 0)

In [24]:
viewer = napari.Viewer()
viewer.add_image(actin_img, colormap = 'red', blending = 'additive')
mito_layer = viewer.add_image(mito_img, colormap = 'green', blending = 'additive')
lipid_layer = viewer.add_image(lipid_img, colormap = 'magenta', blending = 'additive')
overlap_layer = viewer.add_layer(Labels(overlap_mask))
cell_layer = viewer.add_layer(Labels(cell_mask))

In [25]:
overlap_mask_path = output_path / (img_list[img_index].name.replace('.tif', '_overlap.tif'))
imwrite(overlap_mask_path, overlap_mask)

viewer.close()

In [711]:
#overlap_mask_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_overlap.tif'))
#
#overlap_mask = imread(overlap_mask_path)

### EDT maps--------------------------------------------------

###### Central vein EDT map

In [26]:
viewer = napari.Viewer()
viewer.add_image(actin_img, colormap = 'red', blending = 'additive')
mito_layer = viewer.add_image(mito_img, colormap = 'green', blending = 'additive')
lipid_layer = viewer.add_image(lipid_img, colormap = 'magenta', blending = 'additive')

In [28]:
cv_mask = viewer.layers['Labels'].data
cv_mask = np.where(cv_mask == 1, 0, 1)
cv_edt = nd.distance_transform_edt(cv_mask)

In [29]:
cv_edt_layer = viewer.add_image(cv_edt, contrast_limits = [0, 65535])

In [30]:
cv_edt_path = output_path / (img_list[img_index].name.replace('.tif', '_cv_edt.tif'))
imwrite(cv_edt_path, cv_edt)

viewer.close()

###### Portal vein EDT map

In [31]:
viewer = napari.Viewer()
viewer.add_image(actin_img, colormap = 'red', blending = 'additive')
mito_layer = viewer.add_image(mito_img, colormap = 'green', blending = 'additive')
lipid_layer = viewer.add_image(lipid_img, colormap = 'magenta', blending = 'additive')

In [32]:
pv_mask = viewer.layers['Labels'].data
pv_mask = np.where(pv_mask == 1, 0, 1)
pv_edt = nd.distance_transform_edt(pv_mask)

In [33]:
pv_edt_layer = viewer.add_image(pv_edt, contrast_limits = [0, 65535])

In [34]:
pv_edt_path = output_path / (img_list[img_index].name.replace('.tif', '_pv_edt.tif'))
imwrite(pv_edt_path, pv_edt)

viewer.close()

###### Mitochondria EDT map

In [35]:
mito_binary = np.where(mito_mask == 0, 1, 0)
mito_edt = nd.distance_transform_edt(mito_binary)

In [36]:
viewer = napari.Viewer()
viewer.add_image(actin_img, colormap = 'red', blending = 'additive')
mito_layer = viewer.add_image(mito_img, colormap = 'green', blending = 'additive')
lipid_layer = viewer.add_image(lipid_img, colormap = 'magenta', blending = 'additive')
mito_edt_layer = viewer.add_image(mito_edt)

In [37]:
mito_edt_path = output_path / (img_list[img_index].name.replace('.tif', '_mito_edt.tif'))
imwrite(mito_edt_path, mito_edt)

viewer.close()

###### Lipid EDT map

In [38]:
lipid_binary = np.where(lipid_mask == 0, 1, 0)
lipid_edt = nd.distance_transform_edt(lipid_binary)

In [39]:
viewer = napari.Viewer()
viewer.add_image(actin_img, colormap = 'red', blending = 'additive')
mito_layer = viewer.add_image(mito_img, colormap = 'green', blending = 'additive')
lipid_layer = viewer.add_image(lipid_img, colormap = 'magenta', blending = 'additive')
lipid_edt_layer = viewer.add_image(lipid_edt)

In [40]:
lipid_edt_path = output_path / (img_list[img_index].name.replace('.tif', '_lipid_edt.tif'))
imwrite(lipid_edt_path, lipid_edt)

viewer.close()

### Quantification----------------------------------------------------

<a id='quantification'></a>

###### Portal to central vein distance

In [41]:
pv_df_tmp = roi_quant(pv_mask, cell_mask, mito_img, lipid_img, cv_edt, pv_edt, mito_edt)

  0%|          | 0/1 [00:00<?, ?it/s]C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:78: FutureWarning: `RegionProperties.mean_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_mean` instead. 
  cell_id.append(round(cellprops[roi].mean_intensity))
C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:88: FutureWarning: `RegionProperties.mean_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_mean` instead. 
  mito_int.append(mitoprops[roi].mean_intensity)
C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:89: FutureWarning: `RegionProperties.mean_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_mean` instead. 
  lipid_int.append(lipidprops[roi].mean_intensity)
C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:90: FutureWarning: `RegionProperties

In [42]:
pv_cv_dist = pv_df_tmp['cv_max_dist'][0]

print(pv_cv_dist)

4905.75702618872


###### Cell statistics

In [43]:
cell_df_tmp = roi_quant(cell_mask, cell_mask, mito_img, lipid_img, cv_edt, pv_edt, mito_edt)

  0%|          | 0/111 [00:00<?, ?it/s]C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:78: FutureWarning: `RegionProperties.mean_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_mean` instead. 
  cell_id.append(round(cellprops[roi].mean_intensity))
C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:88: FutureWarning: `RegionProperties.mean_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_mean` instead. 
  mito_int.append(mitoprops[roi].mean_intensity)
C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:89: FutureWarning: `RegionProperties.mean_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_mean` instead. 
  lipid_int.append(lipidprops[roi].mean_intensity)
C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:90: FutureWarning: `RegionProperti

In [44]:
cell_df = pd.DataFrame()
cell_df['cell_id'] = cell_df_tmp['roi_id']
cell_df['cell_area'] = cell_df_tmp['area']
cell_df['cell_centroid_x'] = cell_df_tmp['centroid_x']
cell_df['cell_centroid_y'] = cell_df_tmp['centroid_y']
cell_df['cell_mito_int'] = cell_df_tmp['mito_int']
cell_df['cell_lipid_int'] = cell_df_tmp['lipid_int']
cell_df['cell_cv_min_dist'] = cell_df_tmp['cv_min_dist']
cell_df['cell_cv_max_dist'] = cell_df_tmp['cv_max_dist']
cell_df['cell_cv_mean_dist'] = cell_df_tmp['cv_mean_dist']
cell_df['cell_pv_min_dist'] = cell_df_tmp['pv_min_dist']
cell_df['cell_pv_max_dist'] = cell_df_tmp['pv_max_dist']
cell_df['cell_pv_mean_dist'] = cell_df_tmp['pv_mean_dist']

print(cell_df.head())
print(cell_df.shape)

   cell_id  cell_area  cell_centroid_x  cell_centroid_y  cell_mito_int  \
0        1    17902.0        54.822254        90.616076      61.034633   
1        2    18792.0       182.873776        49.236696      67.059493   
2        3    23509.0       384.784848        52.190182      62.276150   
3        4    34080.0       553.108627       110.753580      64.994894   
4        5    76433.0       938.906323       106.704826      49.312614   

   cell_lipid_int  cell_cv_min_dist  cell_cv_max_dist  cell_cv_mean_dist  \
0        6.814490       4746.166980       4903.540048        4834.531567   
1        7.081950       4624.277349       4841.144493        4716.819829   
2        8.334085       4391.734623       4631.479245        4518.601994   
3        8.229959       4220.339797       4460.866732        4341.988437   
4        7.619902       3787.384586       4192.059756        3966.172543   

   cell_pv_min_dist  cell_pv_max_dist  cell_pv_mean_dist  
0        576.000000        752.811397  

###### Mitochondria statistics

In [45]:
mito_df_tmp = roi_quant(mito_mask, cell_mask, mito_img, lipid_img, cv_edt, pv_edt, lipid_edt)

  0%|          | 0/10522 [00:00<?, ?it/s]C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:78: FutureWarning: `RegionProperties.mean_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_mean` instead. 
  cell_id.append(round(cellprops[roi].mean_intensity))
C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:88: FutureWarning: `RegionProperties.mean_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_mean` instead. 
  mito_int.append(mitoprops[roi].mean_intensity)
C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:89: FutureWarning: `RegionProperties.mean_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_mean` instead. 
  lipid_int.append(lipidprops[roi].mean_intensity)
C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:90: FutureWarning: `RegionProper

In [46]:
mito_df = mito_df_tmp.merge(cell_df, how = 'outer', on = 'cell_id')
mito_df['organelle'] = 'mito'
print(mito_df.head())
print(mito_df.shape)

   roi_id  cell_id   area  perimeter  eccentricity  solidity  \
0     6.0        0  182.0  51.213203      0.804441  0.962963   
1    18.0        0  124.0  44.970563      0.893674  0.925373   
2    19.0        0  109.0  36.727922      0.405962  0.939655   
3    30.0        0  101.0  36.142136      0.566906  0.918182   
4    31.0        0   60.0  28.485281      0.846456  0.937500   

   axis_major_length  axis_minor_length  feret_diameter_max   centroid_x  ...  \
0          20.014251          11.889124           20.615528   610.895604  ...   
1          18.939580           8.498509           19.235384  1762.491935  ...   
2          12.339134          11.276612           13.416408  1807.339450  ...   
3          12.515745          10.310254           13.601471  4875.950495  ...   
4          12.021511           6.400951           12.165525     3.350000  ...   

   cell_centroid_y  cell_mito_int  cell_lipid_int  cell_cv_min_dist  \
0              NaN            NaN             NaN        

###### Lipid statistics

In [47]:
lipid_df_tmp = roi_quant(lipid_mask, cell_mask, mito_img, lipid_img, cv_edt, pv_edt, mito_edt)

  0%|          | 0/2685 [00:00<?, ?it/s]C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:78: FutureWarning: `RegionProperties.mean_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_mean` instead. 
  cell_id.append(round(cellprops[roi].mean_intensity))
C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:88: FutureWarning: `RegionProperties.mean_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_mean` instead. 
  mito_int.append(mitoprops[roi].mean_intensity)
C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:89: FutureWarning: `RegionProperties.mean_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_mean` instead. 
  lipid_int.append(lipidprops[roi].mean_intensity)
C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:90: FutureWarning: `RegionPropert

In [48]:
lipid_df = lipid_df_tmp.merge(cell_df, how = 'outer', on = 'cell_id')
lipid_df['organelle'] = 'lipid'
print(lipid_df.head())
print(lipid_df.shape)

   roi_id  cell_id   area  perimeter  eccentricity  solidity  \
0    11.0        0   97.0  33.556349      0.433029  0.979798   
1    12.0        0  388.0  81.698485      0.895581  0.965174   
2    18.0        0  113.0  39.213203      0.648346  0.933884   
3    21.0        0   81.0  30.142136      0.193892  0.987805   
4    22.0        0   72.0  30.142136      0.524061  0.900000   

   axis_major_length  axis_minor_length  feret_diameter_max   centroid_x  ...  \
0          11.697397          10.543802           12.083046  4311.237113  ...   
1          33.989672          15.121972           34.365681  4885.425258  ...   
2          13.907771          10.588620           15.297059  4745.477876  ...   
3          10.251167          10.056630           10.770330  3650.777778  ...   
4          10.427379           8.880795           11.180340  3714.402778  ...   

   cell_centroid_y  cell_mito_int  cell_lipid_int  cell_cv_min_dist  \
0              NaN            NaN             NaN        

###### Overlap statistics

In [49]:
overlap_df_tmp = roi_quant(overlap_mask, cell_mask, mito_img, lipid_img, cv_edt, pv_edt, mito_edt)

  0%|          | 0/107 [00:00<?, ?it/s]C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:78: FutureWarning: `RegionProperties.mean_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_mean` instead. 
  cell_id.append(round(cellprops[roi].mean_intensity))
C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:88: FutureWarning: `RegionProperties.mean_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_mean` instead. 
  mito_int.append(mitoprops[roi].mean_intensity)
C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:89: FutureWarning: `RegionProperties.mean_intensity` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.intensity_mean` instead. 
  lipid_int.append(lipidprops[roi].mean_intensity)
C:\Users\kxll\AppData\Local\Temp\ipykernel_5468\743234405.py:90: FutureWarning: `RegionProperti

In [50]:
overlap_df = pd.DataFrame() 
overlap_df['cell_id'] = overlap_df_tmp['roi_id']
overlap_df['overlap_area'] = overlap_df_tmp['area']

print(overlap_df.head())
print(overlap_df.shape)

   cell_id  overlap_area
0        1         185.0
1        2          11.0
2        3         176.0
3        4         313.0
4        5         398.0
(107, 2)


###### Combine dataframes

In [51]:
df = pd.concat([mito_df, lipid_df])
df = df.merge(overlap_df, how = 'outer', on = 'cell_id')
df['pv_cv_dist'] = pv_cv_dist

print(df.head())
print(df.shape)

   roi_id  cell_id   area  perimeter  eccentricity  solidity  \
0     6.0        0  182.0  51.213203      0.804441  0.962963   
1    18.0        0  124.0  44.970563      0.893674  0.925373   
2    19.0        0  109.0  36.727922      0.405962  0.939655   
3    30.0        0  101.0  36.142136      0.566906  0.918182   
4    31.0        0   60.0  28.485281      0.846456  0.937500   

   axis_major_length  axis_minor_length  feret_diameter_max   centroid_x  ...  \
0          20.014251          11.889124           20.615528   610.895604  ...   
1          18.939580           8.498509           19.235384  1762.491935  ...   
2          12.339134          11.276612           13.416408  1807.339450  ...   
3          12.515745          10.310254           13.601471  4875.950495  ...   
4          12.021511           6.400951           12.165525     3.350000  ...   

   cell_lipid_int  cell_cv_min_dist  cell_cv_max_dist  cell_cv_mean_dist  \
0             NaN               NaN               Na

In [52]:
df_path = output_path / (img_list[img_index].name.replace('.tif', '_output.csv'))
df.to_csv(df_path, header = True)

[Load another image](#load_cell) [Set new path](#set_path) [New mito seg](#mito_path) [New lipid seg](#lipid_path)

### Test---------------------------------

In [75]:
df_tmp = pd.DataFrame()
df_tmp['relative_cell_cv_dist'] = df['cell_cv_max_dist'] / df['pv_cv_dist']
    
conditions = [
    df_tmp['relative_cell_cv_dist'].gt(11/12),
    df_tmp['relative_cell_cv_dist'].gt(10/12) & df_tmp['relative_cell_cv_dist'].le(11/12),
    df_tmp['relative_cell_cv_dist'].gt(9/12) & df_tmp['relative_cell_cv_dist'].le(10/12),
    df_tmp['relative_cell_cv_dist'].gt(8/12) & df_tmp['relative_cell_cv_dist'].le(9/12),
    df_tmp['relative_cell_cv_dist'].gt(7/12) & df_tmp['relative_cell_cv_dist'].le(8/12),
    df_tmp['relative_cell_cv_dist'].gt(6/12) & df_tmp['relative_cell_cv_dist'].le(7/12),
    df_tmp['relative_cell_cv_dist'].gt(5/12) & df_tmp['relative_cell_cv_dist'].le(6/12),
    df_tmp['relative_cell_cv_dist'].gt(4/12) & df_tmp['relative_cell_cv_dist'].le(5/12),
    df_tmp['relative_cell_cv_dist'].gt(3/12) & df_tmp['relative_cell_cv_dist'].le(4/12),
    df_tmp['relative_cell_cv_dist'].gt(2/12) & df_tmp['relative_cell_cv_dist'].le(3/12),
    df_tmp['relative_cell_cv_dist'].gt(1/12) & df_tmp['relative_cell_cv_dist'].le(2/12),
    df_tmp['relative_cell_cv_dist'].ge(0) & df_tmp['relative_cell_cv_dist'].le(1/12)
]
    
choices = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
    
df_tmp['ring'] = np.select(conditions, choices)
df_tmp['roi_id'] = df['cell_id']

print(df_tmp.shape)
print(df_tmp.head())

(7413, 3)
   relative_cell_cv_dist  ring  roi_id
0               0.976487     1       1
1               0.976487     1       1
2               0.976487     1       1
3               0.976487     1       1
4               0.976487     1       1


In [78]:
roi_id = cell_mask.reshape(-1)
mask_df = pd.DataFrame()
mask_df['roi_id'] = roi_id
mask_df = pd.merge(mask_df, df_tmp, how = 'left', on = ['roi_id'])

print(mask_df.shape)
print(mask_df.head())

(1207298171, 3)
   roi_id  relative_cell_cv_dist  ring
0       0                    NaN     0
1       0                    NaN     0
2       0                    NaN     0
3       0                    NaN     0
4       0                    NaN     0


In [80]:
print(cell_mask.shape)
roi_id = cell_mask.reshape(-1)
print(len(roi_id))

(1452, 5208)
7562016


In [81]:
mask_df = pd.DataFrame()
mask_df['roi_id'] = roi_id

print(mask_df.shape)

(7562016, 1)


In [ ]:
mask_df2 = pd.merge(mask_df, df_tmp, how = 'left', on =['roi_id'])

print(mask_df2.shape)

In [84]:
print(df_tmp.head())

   relative_cell_cv_dist  ring  roi_id
0               0.976487     1       1
1               0.976487     1       1
2               0.976487     1       1
3               0.976487     1       1
4               0.976487     1       1


In [ ]:
mask = mask_df['ring'].to_numpy()
mask = mask.reshape(cell_mask.shape)
mask = np.uint16(mask)

### Load segmentation

###### Using existing segmentation files

<a id='alt_function'></a>

### Set paths

<a id='alt_set_path'></a>

In [ ]:
# Define file paths--------------------------------------------------------------------------

input_path = input('Input path: ')
output_path = input('Output path: ')

In [ ]:
input_path = os.path.normpath(input_path)
output_path = os.path.normpath(output_path)

### Load image

<a id='alt_load_image'></a>

In [ ]:
# Load image-----------------------------------------------------------------------------------------------
tmp_list = os.listdir(input_path)

img_list = [] 
for tmp in tmp_list:
    if re.search('.tif', tmp):
        img_list.append(tmp)
        
print(img_list)
print(len(img_list))
img_index = input('Select image: ')
img_index = int(img_index) - 1
print(img_list[img_index])

In [ ]:
img_path = str(img_list[img_index])
img = imread(img_path)

actin_ch_number = int(input('Input actin channel: ')) - 1
mito_ch_number = int(input('Input mitochondria channel: ')) - 1
lipid_ch_number = int(input('Input lipid channel: ')) - 1

In [ ]:
actin_img = img[actin_ch_number, :, :]
mito_img = img[mito_ch_number, :, :]
lipid_img = img[lipid_ch_number, :, :]

In [ ]:
cell_mask_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_cell.tif'))
cell_mask = imread(cell_mask_path)

mito_mask_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_mito.tif'))
mito_mask = imread(mito_mask_path)

lipid_mask_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_lipid.tif'))
lipid_mask = imread(lipid_mask_path)

cv_edt_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_cv_edt.tif'))
cv_edt = imread(cv_edt_path)

pv_edt_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_pv_edt.tif'))
pv_edt = imread(pv_edt_path)

pv_mask = np.where(pv_edt == 0, 1, 0)

mito_edt_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_mito_edt.tif'))
mito_edt = imread(mito_edt_path)

lipid_edt_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_lipid_edt.tif'))
lipid_edt = imread(lipid_edt_path)

[Go to quantification](#quantification)

### Remake mitochondria segmentation

### Set paths

In [3]:
# Define model paths--------------------------------------------------------------------------------------

cell_model_path = input('Cell model path: ')
mito_model_path = input('Mito model path: ')
lipid_model_path = input('Lipid model path: ')

Cell model path:  Z:/Analysis/[NCI] [TGIMB] Natalie Porat-Shliom/Lauryn Brown/training/cell/models/brownl_cell_01
Mito model path:  Z:/Analysis/[NCI] [TGIMB] Natalie Porat-Shliom/Lauryn Brown/training/mito/models/brownl_mito_03
Lipid model path:  Z:/Analysis/[NCI] [TGIMB] Natalie Porat-Shliom/Lauryn Brown/training/lipid/models/brownl_lipid_01


In [319]:
# Define file paths--------------------------------------------------------------------------

input_path = input('Input path: ')
output_path = input('Output path: ')

Input path:  Z:\Analysis\[NCI] [TGIMB] Natalie Porat-Shliom\Lauryn Brown\images\PLIN5 variants in ND\PLIN5 (1-424)
Output path:  Z:\Analysis\[NCI] [TGIMB] Natalie Porat-Shliom\Lauryn Brown\output\PLIN5 variants in ND\PLIN5 (1-424)


In [320]:
input_path = os.path.normpath(input_path)
output_path = os.path.normpath(output_path)

cell_model_path = os.path.normpath(cell_model_path)
mito_model_path = os.path.normpath(mito_model_path)
lipid_model_path = os.path.normpath(lipid_model_path)

cell_model = models.CellposeModel(gpu = True, pretrained_model = cell_model_path)
mito_model = models.CellposeModel(gpu = True, pretrained_model = mito_model_path)
lipid_model = models.CellposeModel(gpu = True, pretrained_model = lipid_model_path)

### Load image

<a id='mito_path'></a>

In [347]:
# Load image-----------------------------------------------------------------------------------------------
tmp_list = os.listdir(input_path)

img_list = [] 
for tmp in tmp_list:
    if re.search('.tif', tmp):
        img_list.append(tmp)
        
print(img_list)
print(len(img_list))
img_index = input('Select image: ')
img_index = int(img_index) - 1
print(img_list[img_index])

['3916C M1_L1_Cropped PP-PC Axis 1.tif', '3916C M1_L1_Cropped PP-PC Axis 2.tif', '3916C M1_L2_Cropped PP-PC Axis 3.tif', '3916C M1_L2_Cropped PP-PC Axis 4.tif', '3916C M1_L2_Cropped PP-PC Axis 5.tif', '3916C M1_L3_Cropped PP-PC Axis 6.tif', '3917A M2_L1_Cropped PP-PC Axis 1.tif', '3917A M2_L1_Cropped PP-PC Axis 2.tif', '3917A M2_L2_Cropped PP-PC Axis 3.tif', '3917A M2_L2_Cropped PP-PC Axis 4.tif', '3917A M2_L3_Cropped PP-PC Axis 5.tif', '3917A M2_L3_Cropped PP-PC Axis 6.tif']
12


Select image:  2


3916C M1_L1_Cropped PP-PC Axis 2.tif


In [348]:
img_path = os.path.join(input_path, img_list[img_index])
img = imread(img_path)

print(img.shape)

viewer = napari.view_image(img)

(3, 1437, 5196)


In [349]:
actin_ch_number = int(input('Input actin channel: ')) - 1
mito_ch_number = int(input('Input mitochondria channel: ')) - 1
lipid_ch_number = int(input('Input lipid channel: ')) - 1

viewer.close()

Input actin channel:  3
Input mitochondria channel:  1
Input lipid channel:  2


In [350]:
actin_img = img[actin_ch_number, :, :]
mito_img = img[mito_ch_number, :, :]
lipid_img = img[lipid_ch_number, :, :]

In [351]:
cell_mask_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_cell.tif'))
cell_mask = imread(cell_mask_path)

#mito_mask_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_mito.tif'))
#mito_mask = imread(mito_mask_path)

lipid_mask_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_lipid.tif'))
lipid_mask = imread(lipid_mask_path)

cv_edt_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_cv_edt.tif'))
cv_edt = imread(cv_edt_path)

pv_edt_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_pv_edt.tif'))
pv_edt = imread(pv_edt_path)

pv_mask = np.where(pv_edt == 0, 1, 0)

#mito_edt_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_mito_edt.tif'))
#mito_edt = imread(mito_edt_path)

lipid_edt_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_lipid_edt.tif'))
lipid_edt = imread(lipid_edt_path)

### Mitochondria segmentation

In [352]:
mito_mask = model_apply(mito_model, mito_img)
mito_mask = np.uint16(mito_mask)

In [353]:
viewer = napari.view_image(mito_img, colormap = 'green', blending = 'additive')
mito_layer = viewer.add_labels(mito_mask)

In [354]:
mito_mask_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_mito.tif'))
imwrite(mito_mask_path, mito_mask)

viewer.close()

### Overlap

In [355]:
mito_binary = np.where(mito_mask > 0, 1, 0)
lipid_binary = np.where(lipid_mask > 0 , 1, 0)

overlap_tmp = np.add(mito_binary, lipid_binary)
overlap_mask = np.where(overlap_tmp == 2, cell_mask, 0)

In [356]:
viewer = napari.view_image(actin_img, colormap = 'red', blending = 'additive')
mito_layer = viewer.add_image(mito_img, colormap = 'green', blending = 'additive')
lipid_layer = viewer.add_image(lipid_img, colormap = 'magenta', blending = 'additive')
overlap_layer = viewer.add_labels(overlap_mask)
cell_layer = viewer.add_labels(cell_mask)

In [357]:
overlap_mask_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_overlap.tif'))
imwrite(overlap_mask_path, overlap_mask)

viewer.close()

###### Mitochondria EDT map

In [358]:
mito_binary = np.where(mito_mask == 0, 1, 0)
mito_edt = nd.distance_transform_edt(mito_binary)

In [359]:
viewer = napari.view_image(actin_img, colormap = 'red', blending = 'additive')
mito_layer = viewer.add_image(mito_img, colormap = 'green', blending = 'additive')
lipid_layer = viewer.add_image(lipid_img, colormap = 'magenta', blending = 'additive')
mito_edt_layer = viewer.add_image(mito_edt)

In [360]:
mito_edt_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_mito_edt.tif'))
imwrite(mito_edt_path, mito_edt)

viewer.close()

[Go to quantification](#quantification)

### Remake lipid segmentation

### Set paths

In [3]:
# Define model paths--------------------------------------------------------------------------------------

cell_model_path = input('Cell model path: ')
mito_model_path = input('Mito model path: ')
lipid_model_path = input('Lipid model path: ')

Cell model path:  Z:/Analysis/[NCI] [TGIMB] Natalie Porat-Shliom/Lauryn Brown/training/cell/models/brownl_cell_01
Mito model path:  Z:/Analysis/[NCI] [TGIMB] Natalie Porat-Shliom/Lauryn Brown/training/mito/models/brownl_mito_03
Lipid model path:  Z:/Analysis/[NCI] [TGIMB] Natalie Porat-Shliom/Lauryn Brown/training/lipid/models/brownl_lipid_01


In [656]:
# Define file paths--------------------------------------------------------------------------

input_path = input('Input path: ')
output_path = input('Output path: ')

Input path:  Z:\Analysis\[NCI] [TGIMB] Natalie Porat-Shliom\Lauryn Brown\images\PLIN5 variants in ND\PLIN5E (1-424)
Output path:  Z:\Analysis\[NCI] [TGIMB] Natalie Porat-Shliom\Lauryn Brown\output\PLIN5 variants in ND\PLIN5E (1-424)


In [657]:
input_path = os.path.normpath(input_path)
output_path = os.path.normpath(output_path)

cell_model_path = os.path.normpath(cell_model_path)
mito_model_path = os.path.normpath(mito_model_path)
lipid_model_path = os.path.normpath(lipid_model_path)

cell_model = models.CellposeModel(gpu = True, pretrained_model = cell_model_path)
mito_model = models.CellposeModel(gpu = True, pretrained_model = mito_model_path)
lipid_model = models.CellposeModel(gpu = True, pretrained_model = lipid_model_path)

### Load image

<a id='lipid_path'></a>

In [955]:
# Load image-----------------------------------------------------------------------------------------------
tmp_list = os.listdir(input_path)

img_list = [] 
for tmp in tmp_list:
    if re.search('.tif', tmp):
        img_list.append(tmp)
        
print(img_list)
print(len(img_list))
img_index = input('Select image: ')
img_index = int(img_index) - 1
print(img_list[img_index])

['3907C M1_L1_Cropped PP-PC Axis 1.tif', '3907C M1_L2_Cropped PP-PC Axis 2.tif', '3907C M1_L2_Cropped PP-PC Axis 3.tif', '3907C M1_L2_Cropped PP-PC Axis 4.tif', '3907C M1_L3_Cropped PP-PC Axis 5.tif', '3907C M1_L3_Cropped PP-PC Axis 6.tif', '3920E M2_L1_Cropped PP-Pc Axis 1.tif', '3920E M2_L1_Cropped PP-PC Axis 2.tif', '3920E M2_L1_Cropped PP-PC Axis 3.tif', '3920E M2_L1_Cropped PP-PC Axis 4.tif', '3920E M2_L2_Cropped PP-PC Axis 5.tif', '3920E M2_L2_Cropped PP-PC Axis 6.tif']
12


Select image:  12


3920E M2_L2_Cropped PP-PC Axis 6.tif


In [956]:
img_path = os.path.join(input_path, img_list[img_index])
img = imread(img_path)

print(img.shape)

viewer = napari.view_image(img)

(3, 1440, 5184)


In [957]:
actin_ch_number = int(input('Input actin channel: ')) - 1
mito_ch_number = int(input('Input mitochondria channel: ')) - 1
lipid_ch_number = int(input('Input lipid channel: ')) - 1

viewer.close()

Input actin channel:  3
Input mitochondria channel:  1
Input lipid channel:  2


In [958]:
actin_img = img[actin_ch_number, :, :]
mito_img = img[mito_ch_number, :, :]
lipid_img = img[lipid_ch_number, :, :]

In [959]:
cell_mask_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_cell.tif'))
cell_mask = imread(cell_mask_path)

mito_mask_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_mito.tif'))
mito_mask = imread(mito_mask_path)

#lipid_mask_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_lipid.tif'))
#lipid_mask = imread(lipid_mask_path)

cv_edt_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_cv_edt.tif'))
cv_edt = imread(cv_edt_path)

pv_edt_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_pv_edt.tif'))
pv_edt = imread(pv_edt_path)

pv_mask = np.where(pv_edt == 0, 1, 0)

mito_edt_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_mito_edt.tif'))
mito_edt = imread(mito_edt_path)

#lipid_edt_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_lipid_edt.tif'))
#lipid_edt = imread(lipid_edt_path)

### Lipid segmentation

In [960]:
lipid_mask_tmp = model_apply(lipid_model, lipid_img)
lipid_mask_tmp = np.uint16(lipid_mask_tmp)

lipid_df_tmp = object_quant(lipid_mask_tmp, lipid_img)
lipid_mask = object_filter(lipid_mask_tmp, lipid_df_tmp, 20)

100%|███████████████████████████████████████████████████████████████████████████| 1551/1551 [00:00<00:00, 41932.23it/s]


In [961]:
viewer = napari.view_image(lipid_img, colormap = 'magenta', blending = 'additive')
lipid_layer = viewer.add_labels(lipid_mask)

In [962]:
lipid_mask = viewer.layers['lipid_mask'].data

In [963]:
lipid_mask_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_lipid.tif'))
imwrite(lipid_mask_path, lipid_mask)

viewer.close()

### Overlap

In [964]:
mito_binary = np.where(mito_mask > 0, 1, 0)
lipid_binary = np.where(lipid_mask > 0 , 1, 0)

overlap_tmp = np.add(mito_binary, lipid_binary)
overlap_mask = np.where(overlap_tmp == 2, cell_mask, 0)

In [965]:
viewer = napari.view_image(actin_img, colormap = 'red', blending = 'additive')
mito_layer = viewer.add_image(mito_img, colormap = 'green', blending = 'additive')
lipid_layer = viewer.add_image(lipid_img, colormap = 'magenta', blending = 'additive')
overlap_layer = viewer.add_labels(overlap_mask)
cell_layer = viewer.add_labels(cell_mask)

In [966]:
overlap_mask_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_overlap.tif'))
imwrite(overlap_mask_path, overlap_mask)

viewer.close()

###### Lipid EDT map

In [967]:
lipid_binary = np.where(lipid_mask == 0, 1, 0)
lipid_edt = nd.distance_transform_edt(lipid_binary)

In [968]:
viewer = napari.view_image(actin_img, colormap = 'red', blending = 'additive')
mito_layer = viewer.add_image(mito_img, colormap = 'green', blending = 'additive')
lipid_layer = viewer.add_image(lipid_img, colormap = 'magenta', blending = 'additive')
lipid_edt_layer = viewer.add_image(lipid_edt)

In [969]:
lipid_edt_path = os.path.join(output_path, img_list[img_index].replace('.tif', '_lipid_edt.tif'))
imwrite(lipid_edt_path, lipid_edt)

viewer.close()

[Go to quantification](#quantification)